# dy/dx는 왜 분수처럼 작동하나

> 미적분 12강 · dy/dx의 비밀 (마지막 강의)

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [dy/dx는 왜 분수처럼 작동하나](https://mioon1402.github.io/timeseriesdata/calc/C12-dydx.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 무엇이 문제인가

## 1. 무한소의 유령 — 라이프니츠의 꿈

## 2. 20세기의 반전 — 접선 위의 진짜 좌표

## 3. 라이프니츠 표기법이 300년을 살아남은 이유

## 4. 부록 · 부분적분법의 기하학적 정체

## 5. 파이썬으로 확인하기

**12-1. 접선 위에서는 dy = f′(x)dx 가 정확하다**

In [ ]:
import numpy as np

f  = lambda x: x**2
fp = lambda x: 2*x
x0 = 1.0

print(f"{'dx':>8} {'dy (접선 위)':>16} {'Δy (곡선 위)':>16} {'차':>12}")
for dx in [1.0, 0.5, 0.1, 0.01]:
    dy = fp(x0) * dx                      # 접선 위 — 정의상 정확
    Δy = f(x0 + dx) - f(x0)               # 곡선 위 — 실제 변화
    print(f"{dx:>8} {dy:>16.8f} {Δy:>16.8f} {Δy-dy:>12.8f}")

print("\n→ dy 와 Δy 는 다르다. 차이는 정확히 dx² (4강의 '두 번 작은 것').")
print("  그런데 dy/dx 는 dx 가 무엇이든 언제나 정확히 f′(x₀) = 2 다.")
for dx in [1.0, 0.5, 0.1, 0.01]:
    print(f"    dx={dx:<6} dy/dx = {fp(x0)*dx/dx}")

**12-2. 연쇄법칙의 du 약분이 진짜인지**

In [ ]:
import numpy as np

g  = lambda x: x**3          # u = g(x)
gp = lambda x: 3*x**2
h  = np.sin                  # y = h(u)
hp = np.cos

x, dx = 1.1, 0.001

du = gp(x) * dx              # 접선 위에서의 u 변화
dy = hp(g(x)) * du           # 접선 위에서의 y 변화

print(f"dx = {dx}")
print(f"du = g′(x)·dx      = {du:.10f}")
print(f"dy = h′(g(x))·du   = {dy:.10f}")
print(f"dy/dx              = {dy/dx:.10f}")
print(f"(dy/du)·(du/dx)    = {(dy/du)*(du/dx):.10f}")
print(f"h′(g(x))·g′(x)     = {hp(g(x))*gp(x):.10f}")
print("\n→ du 가 실제로 약분된다. 눈속임이 아니라 유한한 수들의 진짜 나눗셈이다.")

**12-3. 부분적분 — 두 영토의 넓이**

In [ ]:
import numpy as np
from scipy.integrate import quad

# u = x, v = e^x,  x: 0 → 1
u,  v  = lambda x: x,        np.exp
up, vp = lambda x: 1.0,      np.exp
a, b = 0.0, 1.0

A = quad(lambda x: u(x) * vp(x), a, b)[0]      # ∫u dv  (세로로 쌓은 영토)
B = quad(lambda x: v(x) * up(x), a, b)[0]      # ∫v du  (가로로 쌓은 영토)
uv = u(b)*v(b) - u(a)*v(a)

print(f"∫u dv (왼쪽 영토)   = {A:.10f}")
print(f"∫v du (아래 영토)   = {B:.10f}")
print(f"두 영토의 합        = {A+B:.10f}")
print(f"u(b)v(b)-u(a)v(a)   = {uv:.10f}")
print(f"\n∫u dv = uv - ∫v du = {uv - B:.10f}   ← A 와 같다")
print("\n손으로: ∫₀¹ x·eˣ dx = [x·eˣ]₀¹ - ∫₀¹ eˣ dx = e - (e-1) = 1")

**12-4. 부분적분이 실제로 유용한 예**

In [ ]:
import numpy as np
from scipy.integrate import quad

예제 = [
    ("∫₀¹ x·eˣ dx",     lambda x: x*np.exp(x),      1.0),
    ("∫₁ᵉ ln x dx",     np.log,                     1.0),
    ("∫₀^π x·sin x dx", lambda x: x*np.sin(x),      np.pi),
    ("∫₀¹ x²·eˣ dx",    lambda x: x**2*np.exp(x),   np.e - 2),
]
구간 = [(0,1), (1,np.e), (0,np.pi), (0,1)]

for (이름, f, 손), (a, b) in zip(예제, 구간):
    print(f"{이름:>18}  수치 {quad(f,a,b)[0]:.10f}   손계산 {손:.10f}")

print("\n→ ∫ln x dx 는 u=ln x, v=x 로 놓으면 ∫v du = ∫1 dx 가 되어 순식간에 풀린다.")
print("  '어려운 쪽을 쉬운 쪽으로 맞바꾸는' 전략이다.")

**12-5. 변수분리 — dy 와 dx 를 실제로 옮겨본다**

In [ ]:
import sympy as sp

x = sp.Symbol('x')
y = sp.Function('y')

# dy/dx = k·y  →  dy/y = k dx  →  ln y = kx + C  →  y = y0 e^(kx)
k = sp.Symbol('k')
해 = sp.dsolve(sp.Eq(y(x).diff(x), k*y(x)), y(x))
print("dy/dx = k·y 의 해 :", 해)
print()

# dy/dx = x/y  →  y dy = x dx  →  y²/2 = x²/2 + C  →  y² - x² = 상수 (쌍곡선)
해2 = sp.dsolve(sp.Eq(y(x).diff(x), x/y(x)), y(x))
print("dy/dx = x/y 의 해 :", 해2)
print("\n→ '양변에 dx 를 곱해 넘긴다' 는 조작이 정확한 답을 준다.")
print("  접선 위의 좌표로 보면 이 조작이 왜 정당한지 알 수 있다.")

**12-6. 연습문제**

In [ ]:
# 문제 1. f(x) = √x, x₀ = 4 에서 dx = 0.1 일 때
#         dy (접선 위) 와 Δy (곡선 위) 를 각각 구하고 차이를 보세요.
#         이것이 '미분을 이용한 근사계산' 의 정체입니다. √4.1 ≈ ?

# 문제 2. ∫₀^π x·cos x dx 를 부분적분으로 손으로 풀고 수치로 확인하세요.

# 문제 3. dy/dx = -y/x 를 변수분리로 풀어보세요. (답: xy = 상수)

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
import numpy as np
from scipy.integrate import quad

# 문제 1
x0, dx = 4.0, 0.1
dy = (1/(2*np.sqrt(x0))) * dx
Δy = np.sqrt(x0+dx) - np.sqrt(x0)
print(f"문제 1: dy = {dy:.8f}   Δy = {Δy:.8f}   차 = {Δy-dy:.2e}")
print(f"        √4.1 ≈ 2 + {dy:.6f} = {2+dy:.6f},  참값 {np.sqrt(4.1):.6f}")
print("        공학 계산기가 없던 시절의 '미분을 이용한 근사' 가 이것이다\n")

# 문제 2 — ∫x cos x dx = x sin x + cos x
손 = (np.pi*np.sin(np.pi) + np.cos(np.pi)) - (0 + np.cos(0))
print(f"문제 2: 손계산 [x sin x + cos x]₀^π = {손:.10f}")
print(f"        수치     {quad(lambda x: x*np.cos(x), 0, np.pi)[0]:.10f}\n")

# 문제 3 — dy/y = -dx/x → ln y = -ln x + C → xy = 상수
print("문제 3: dy/y = -dx/x  →  ln|y| = -ln|x| + C  →  xy = 상수")
print("        4강의 '넓이가 보존되는 직사각형' 이 정확히 이 곡선이다!")
for x in [0.5, 1, 2, 4]:
    print(f"        x={x:<4} y=1/x={1/x:<8.4f}  xy={x*(1/x)}")

## 6. 맺으며 — 전체를 한 장의 그림으로

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)